# Data Labeling: Positive Class Mining
Keyword-bootstrapped extraction of hedging language candidates from the clean sentence corpus.

In [1]:
import pandas as pd
import os

In [2]:
os.chdir("..")  # move to project root
df = pd.read_parquet("data/raw/sentences_clean.parquet")
print(f"Loaded {len(df):,} sentences")
print(df.head(3))

Loaded 2,505,229 sentences
  symbol       sector           quarter  \
0    MMM  Industrials  2022_earnings_q1   
1    MMM  Industrials  2022_earnings_q1   
2    MMM  Industrials  2022_earnings_q1   

                                            sentence  
0  John is joining us today to discuss our progre...  
1  During today's conference call, we will be mak...  
2  These statements are based on certain assumpti...  


## Keyword-Bootstrapped Positive Candidate Mining
Flag sentences containing strong hedging indicators across three categories:
epistemic modals, conditional markers, and uncertainty phrases.

In [3]:
import re

HEDGING_PATTERNS = [
    # Epistemic modals
    r'\bmay\b', r'\bmight\b', r'\bcould\b',
    # Conditional markers
    r'\bif\b', r'\bunless\b', r'\bsubject to\b',
    # Uncertainty phrases
    r'\bwe believe\b', r'\bwe expect\b', r'\bwe anticipate\b',
    r'\bwe think\b', r'\blikely\b', r'\bpotentially\b',
    r'\bapproximately\b', r'\buncertain\b', r'\brisk\b',
]

pattern = re.compile('|'.join(HEDGING_PATTERNS), re.IGNORECASE)

df['is_candidate'] = df['sentence'].apply(lambda x: bool(pattern.search(x)))
candidates = df[df['is_candidate']].copy()
print(f"Candidate positives: {len(candidates):,} ({100*len(candidates)/len(df):.1f}%)")

Candidate positives: 330,170 (13.2%)


In [4]:
HEDGING_PATTERNS_STRICT = [
    # Epistemic modals
    r'\bmay\b', r'\bmight\b',
    # Uncertainty phrases — specific enough
    r'\bwe believe\b', r'\bwe expect\b', r'\bwe anticipate\b',
    r'\bwe think\b',
    # Conditional — only the most specific
    r'\bsubject to\b',
    r'\buncertainty\b', r'\buncertain\b',
]

pattern_strict = re.compile('|'.join(HEDGING_PATTERNS_STRICT), re.IGNORECASE)

df['is_candidate_strict'] = df['sentence'].apply(lambda x: bool(pattern_strict.search(x)))
candidates_strict = df[df['is_candidate_strict']].copy()
print(f"Strict candidates: {len(candidates_strict):,} ({100*len(candidates_strict)/len(df):.1f}%)")

Strict candidates: 155,764 (6.2%)


In [5]:
sample = candidates_strict.sample(20, random_state=42)
for _, row in sample.iterrows():
    print(row['sentence'])
    print("---")

But what -- when I think about quarter 3, particularly or quarter 4, we're going to be in a much stronger position on the margin due because of the change in the mix, we think is going to happen.
---
Just wanted your thoughts today on the sort of fragmentation of the Europe service market and maybe how Otis kind of practices might be different there?
---
Fulfillment rates are back to the mid-90s, which is kind of our target, as we have kind of talked in the past, we were low-90s when we think about where we were just a year ago in Q2, actually, we are in high-80s as I am looking at the numbers, and so we have got about 5 points of improvement.
---
First one, a number of your peers have built up pretty sizable deferred fuel balances that may take a couple of years to recover, just given the spike in power and gas prices.
---
As we continue to capitalize on organic and inorganic growth opportunities, we believe that we will greatly expand our scale and deliver long-term growth for our sh

In [6]:
def count_matches(sentence):
    return len(pattern_strict.findall(sentence))

candidates_strict['match_count'] = candidates_strict['sentence'].apply(count_matches)
print(candidates_strict['match_count'].value_counts().sort_index(ascending=False).head(10))

match_count
6         2
5         5
4        34
3       401
2      7132
1    148190
Name: count, dtype: int64


In [7]:
# Sample 1000 high-confidence positives, prioritizing higher match counts
top_candidates = candidates_strict[candidates_strict['match_count'] >= 2].copy()
positives = top_candidates.sample(1000, random_state=42)
print(f"Sampled: {len(positives)} positives")
print(f"Match count distribution:\n{positives['match_count'].value_counts().sort_index(ascending=False)}")

Sampled: 1000 positives
Match count distribution:
match_count
6      1
4      7
3     50
2    942
Name: count, dtype: int64


In [8]:
for _, row in positives.sample(15, random_state=7).iterrows():
    print(row['sentence'])
    print("---")

In closing, while the macroeconomic backdrop has become more uncertain, we believe that we are in the right markets with differentiated products that remain highly valued by our customers.
---
Something that might have been only transmitting one signal, you might have something that's transmitting 400 different signals, and that gets into a higher contact count, more complex connector into that compute, and that's actually areas where we actually drive content increase.
---
Like any statement about the future, these are subject to a number of factors that could cause actual results or events to differ materially from those which we anticipate due to a number of risks and uncertainties.
---
The headwinds we face, whether from foreign exchange, raw and packaging materials and logistics costs or macroeconomic uncertainty are significant, but we believe we are well positioned to deal with these issues.
---
Are there any other, assuming they are ongoing, but is there any others that are imm

In [9]:
positives['label'] = 1
positives = positives[['symbol', 'sector', 'quarter', 'sentence', 'label']]
positives.to_parquet("data/raw/positives.parquet", index=False)
print("Saved.")

Saved.


In [10]:
# Filter out questions and very short sentences
positives_filtered = positives[
    ~positives['sentence'].str.endswith('?') &
    (positives['sentence'].str.len() >= 40)
].copy()

print(f"Before: {len(positives)} → After: {len(positives_filtered)}")

Before: 1000 → After: 847


In [11]:
# Top up to 1000 by sampling additional 2-match candidates not already selected
already_selected = positives_filtered.index
remaining_pool = top_candidates[
    ~top_candidates.index.isin(already_selected) &
    ~top_candidates['sentence'].str.endswith('?') &
    (top_candidates['sentence'].str.len() >= 40)
].copy()

topup = remaining_pool.sample(153, random_state=99)
positives_final = pd.concat([positives_filtered, topup], ignore_index=True)
print(f"Final positive set: {len(positives_final)}")

Final positive set: 1000


In [12]:
# 1. No duplicate sentences
print(f"Duplicates: {positives_final['sentence'].duplicated().sum()}")

# 2. Sector distribution — ensure no single sector dominates
print(positives_final['sector'].value_counts())

# 3. Final random sample of 10 for one last eyeball check
for _, row in positives_final.sample(10, random_state=123).iterrows():
    print(row['sentence'])
    print("---")

Duplicates: 14
sector
Financials                193
Health Care               154
Industrials               135
Information Technology    123
Consumer Discretionary    114
Real Estate                85
Consumer Staples           57
Materials                  52
Communication Services     33
Utilities                  31
Energy                     23
Name: count, dtype: int64
Construction commenced on the 170,000 square foot center in May, and we anticipate Whole Foods opening in 2025.
---
And we may make some decisions where it may be additive to our earnings and it may be accretive to our margin percentage.
---
While we expect the returns from Generative AI to come in over a longer period of time, we’re mapping these investments against the significant monetization opportunities that we expect to be unlocked across customized ad creative, business messaging, a leading AI assistant and organic content generation.
---
And so I think, we may see -- we may enter into an environment where 

In [13]:
positives_final = positives_final.drop_duplicates(subset='sentence').reset_index(drop=True)
print(f"Final count after dedup: {len(positives_final)}")

positives_final['label'] = 1
positives_final = positives_final[['symbol', 'sector', 'quarter', 'sentence', 'label']]
positives_final.to_parquet("data/raw/positives.parquet", index=False)
print("Saved.")

Final count after dedup: 986
Saved.


In [14]:
positives_final.to_csv("data/raw/positives_inspect.csv", index=False)

In [15]:
# Exclude any sentence that matched hedging patterns
true_neg_pool = df[~df['sentence'].str.contains(
    pattern_strict, regex=True
)].copy()

print(f"Pool size: {len(true_neg_pool):,}")

Pool size: 2,349,465


In [16]:
# Filter for likely factual/declarative sentences
FACTUAL_INDICATORS = [
    r'\bincreased\b', r'\bdecreased\b', r'\bgrew\b', r'\bdeclined\b',
    r'\breported\b', r'\bdelivered\b', r'\bachieved\b', r'\bgenerated\b',
    r'\bwas\b', r'\bwere\b', r'\bis\b', r'\bare\b',
    r'\bmargin\b', r'\brevenue\b', r'\bearnings\b', r'\bgrowth\b',
]

factual_pattern = re.compile('|'.join(FACTUAL_INDICATORS), re.IGNORECASE)

true_neg_pool['is_factual'] = true_neg_pool['sentence'].apply(
    lambda x: bool(factual_pattern.search(x))
)

factual_pool = true_neg_pool[true_neg_pool['is_factual']].copy()
print(f"Factual pool size: {len(factual_pool):,}")

Factual pool size: 1,081,523


In [17]:
true_negatives = factual_pool.sample(18600, random_state=42)

# Quick eyeball check
for _, row in true_negatives.sample(10, random_state=7).iterrows():
    print(row['sentence'])
    print("---")

17:19 Reported EPS was $2.92 in the quarter, compared to $3.46 a year ago.
---
On the 733,000 square feet in the quarter and subsequent, it was about 71% existing tenants in the portfolio.
---
At Pentair, we are focused on creating a better world for people and the planet through smart, sustainable water solutions.
---
Finally, any references to earnings per share or EPS made during this conference call refer to diluted earnings per common share.
---
And so during that transition, when we’re accounting for that and our third-party partners are accounting for that, there can be a little bit of an extended drag.
---
We are currently executing our fall planning cycle in alignment with Sempra to assess future capital projects, and we'll be announcing our new five year capital plan on the fourth quarter call.
---
We are raising our 2024 adjusted earnings guidance range to $3.29 to $3.35 per share from $3.27 to $3.33 per share, as Garrick noted, with continued confidence toward the high end 

In [18]:
# Remove questions and short sentences
true_negatives = true_negatives[
    ~true_negatives['sentence'].str.endswith('?') &
    (true_negatives['sentence'].str.len() >= 40)
].copy()

print(f"After filtering: {len(true_negatives):,}")

After filtering: 16,171


In [19]:
already_selected_tn = true_negatives.index
remaining_factual = factual_pool[
    ~factual_pool.index.isin(already_selected_tn) &
    ~factual_pool['sentence'].str.endswith('?') &
    (factual_pool['sentence'].str.len() >= 40)
].copy()

topup_tn = remaining_factual.sample(2429, random_state=55)
true_negatives_final = pd.concat([true_negatives, topup_tn], ignore_index=True)
print(f"Final true negatives: {len(true_negatives_final):,}")

Final true negatives: 18,600


In [20]:
true_negatives_final['label'] = 0
true_negatives_final = true_negatives_final[['symbol', 'sector', 'quarter', 'sentence', 'label']]
true_negatives_final.to_parquet("data/raw/true_negatives.parquet", index=False)
print("Saved.")

Saved.
